# Clasificación de gatos y perros mediante fine tuning de redes convolucionales preentrenadas

### Juan Sebastian Garcia Perez

**Problema:** dada la fotografía de una mascota, determinar automáticamente si es un gato o un perro.

**Dominio:** Visión artificial, clasificación de imágenes de animales domésticos.

**Utilidad:** Automatizar el etiquetado de fotos de mascotas. Un refugio o un portal de adopción recibe miles de fotos y necesita clasificarlas por especie sin que una persona las revise una por una.

**Base de datos:** Cats vs Dogs: 23.262 fotografías a color y 1 variable de salida categórica binaria. BD publicada en [TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs).

**Tipo de problema:** aprendizaje supervisado de clasificación binaria de imágenes.

**Modelos entrenados:** VGG16 y ResNet50 preentrenadas con ImageNet, ajustadas con fine tuning y búsqueda en malla de hiperparámetros.

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print("TensorFlow:", tf.__version__)
print("GPU disponible:", len(tf.config.list_physical_devices("GPU")) > 0)

TensorFlow: 2.19.0
GPU disponible: True


# 2. Descripción de la BD

Cats vs Dogs fue creada por Microsoft Research junto con Petfinder.com, un portal de adopción de mascotas de Estados Unidos. Las fotos son de animales reales de refugios y fueron etiquetadas a mano por el personal de esos refugios. Nació en 2007 para el proyecto Asirra, un CAPTCHA que pedía distinguir gatos de perros porque en ese momento era una tarea fácil para una persona y muy difícil para una máquina.

| Característica | Valor |
|---|---|
| Imágenes | 23.262 fotografías JPEG a color |
| Variable de entrada | image: matriz de píxeles de alto x ancho x 3 canales (RGB), enteros de 0 a 255 |
| Variable de salida | label (binaria) (cat / dog) |
| Tamaño de las imágenes | Variable, entre 4 y 500 píxeles de lado (mediana 467 x 375) |
| Imágenes dañadas | 1.738 de las 25.000 originales, ya eliminadas por TensorFlow Datasets |
| Reparto de clases | 11.658 gatos (50,1 %), 11.604 perros (49,9 %) |
| Particiones de origen | Solo trae train, la separación en entrenamiento, validación y prueba la hacemos nosotros |

Variable de salida (label):

| Aspecto | label |
|---|---|
| Qué es | La especie del animal de la foto |
| Tipo | Categórica nominal (sin orden) y binaria |
| Valores | cat (11.658 imágenes, 50,1 %) y dog (11.604 imágenes, 49,9 %) |
| Codificación | cat -> 0, dog -> 1 (ya viene codificada) |

A diferencia de una base de datos tabular, aquí no hay columnas con significado propio: las variables de entrada son los píxeles, y un píxel suelto no dice nada. Lo que distingue a un gato de un perro son patrones espaciales (orejas, hocico, pelaje), y por eso se usan redes neuronales convolucionales.